In [ ]:
import nltk
import re
import pandas as pd
from nltk.tokenize import sent_tokenize
from nltk.corpus import stopwords
import string
from string import punctuation
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer
from heapq import nlargest

# PREPROCESSING

In [ ]:
# Load the dataset
train = pd.read_csv('E:/sub/NLP/project_dataset/train.csv', encoding='iso-8859-1')
test = pd.read_csv('E:/sub/NLP/project_dataset/eval.csv', encoding='iso-8859-1')

In [ ]:
# Define function for text preprocessing
def preprocess_text(text):
    # Remove non-alphabetic characters
    text = re.sub('[^a-zA-Z]', ' ', text)
    
    # Convert to lowercase
    text = text.lower()
    
    #Remove punctuation
    text="".join([char for char in text if char not in string.punctuation])
    
    #adding start and end tokens
    text = '<start> ' + text + ' <end>'
    
    # Tokenize into sentences
    sentences = sent_tokenize(text)
     # Tokenize the text
    words = word_tokenize(text)
  
    # Join the preprocessed sentences
    preprocessed_sentences = []
    preprocessed_sentences.append(' '.join(words))
    
    # Join the preprocessed sentences
    preprocessed_text = ' '.join(preprocessed_sentences)
    
    return preprocessed_text

# Apply the preprocessing function to the train text and headlines columns
train['input'] = train['input'].apply(preprocess_text)

# Save the preprocessed train dataset
train.to_csv('preprocessed_train.csv', index=False)
# Print the preprocessed train dataset
print(train.head())

print("-----------------------------------------------------------------------------")

# Apply the preprocessing function to the test text and headlines columns
test['input'] = test['input'].apply(preprocess_text)

# Save the preprocessed test dataset
test.to_csv('preprocessed_test.csv', index=False)
# Print the preprocessed test dataset
print(test.head())


# POS

In [ ]:
# Define function for train POS tagging
def pos_tagging(text):
    # Tokenize the text
    tokens = nltk.word_tokenize(text)
    # Perform POS tagging
    pos_tags = nltk.pos_tag(tokens)
    # Extract the POS tags
    pos_tags_only = [tag[1] for tag in pos_tags]
    # Join the POS tags into a string
    pos_tags_string = " ".join(pos_tags_only)
    return pos_tags_string

In [ ]:
# Apply POS tagging to the train text and headlines columns
train['input_POS'] = train['input'].apply(pos_tagging)

# Apply POS tagging to the test text and headlines columns
test['input_POS'] = test['input'].apply(pos_tagging)

In [ ]:
# Save the POS-tagged train dataset
train.to_csv('preprocessed_pos.csv', index=False)

# Save the POS-tagged test dataset
test.to_csv('preprocessed_pos_test.csv', index=False)

In [ ]:
print(train['input_POS'].head())
print("-----------------------------------------------------------------------------")
print(test['input_POS'].head())

# Count Vectorization

In [ ]:
# create a document term matrix
from sklearn.feature_extraction.text import CountVectorizer
cv_count = CountVectorizer()

In [ ]:
X_count = cv_count.fit(train['input'])
# automatically remove any one character word like a
print(X_count.vocabulary_)
# unique words in corpus
print(cv_count.get_feature_names_out())

In [ ]:
# create a document term matrix
X_count = cv_count.transform(train['input'])
print(X_count.shape)
#document term matrix
print(X_count.toarray())
df = pd.DataFrame(X_count.toarray(),columns=cv_count.get_feature_names_out())
print(df)

# N-Grams Vecorization

In [ ]:
#bigram only
cv_N_Grams = CountVectorizer(ngram_range=(2,2))
X_N_Grams = cv_N_Grams.fit_transform(train['input'])
print(X_N_Grams.shape)
#document term matrix
print(X_N_Grams.toarray())
df = pd.DataFrame(X_N_Grams.toarray(),columns=cv_N_Grams.get_feature_names_out())
df

In [ ]:
# mean bigram and trigram
cv_N_Grams = CountVectorizer(ngram_range=(2,3))
X_N_Grams = cv_N_Grams.fit_transform(train['input'])
print(X_N_Grams.shape)
#document term matrix
print(X_N_Grams.toarray())
df = pd.DataFrame(X_N_Grams.toarray(),columns=cv_N_Grams.get_feature_names_out())
df

# TF-IDF Vectorization

In [ ]:
import string
from nltk import word_tokenize
import nltk
from nltk.stem import PorterStemmer
ps = PorterStemmer()
stopwords = nltk.corpus.stopwords.words('english')
def clean_text(txt):
    txt_nopunct = "".join([c for c in txt if c not in string.punctuation])
    tokens = word_tokenize(txt_nopunct)
    txt_clean = [word for word in tokens if word not in stopwords]
    tokens_stem = [ps.stem(word) for word in txt_clean]
    return tokens_stem

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf_vec_train = TfidfVectorizer(analyzer=clean_text)
tfidf_vec_train_fit = tfidf_vec_train.fit(train['input'])
X_tfidf = tfidf_vec_train.fit_transform(train['input'])
print(X_tfidf.shape)
df = pd.DataFrame(X_tfidf.toarray(),columns=tfidf_vec_train.get_feature_names_out())
df.head()

# Feature Engineering

In [ ]:
import nltk
from nltk.tokenize import sent_tokenize
from nltk.corpus import stopwords
# Function to calculate sentence length and complexity
def calculate_sentence_features(text):
    # Tokenize text into sentences
    sentences = sent_tokenize(text)
    
    # Calculate average sentence length
    total_sentence_length = sum(len(sentence.split()) for sentence in sentences)
    avg_sentence_length = total_sentence_length / len(sentences)
    
    # Calculate type-token ratio
    words = nltk.word_tokenize(text)
    words = [word.lower() for word in words if word.isalpha()]
    words = [word for word in words if word not in stopwords.words('english')]
    types = len(set(words))
    tokens = len(words)
    ttr = types / tokens
    
    return avg_sentence_length, ttr


# Example usage of the function
text = "This is a sample sentence. It has multiple clauses and is of average length."
avg_sentence_length, ttr = calculate_sentence_features(text)
print(avg_sentence_length)
print(ttr)

train['input_len'] = train['input'].apply(lambda x: len(x))
train.head()

train['target_len'] = train['target'].apply(lambda x: len(x))
train.head()


# Padding

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Define a list of input text strings
texts = train['input']
# Create a tokenizer and fit on the text data
vocab_size = 10000000
oov_tok = '0' # Out of Vocabulary
tokenizer = Tokenizer(num_words = vocab_size, oov_token=oov_tok)
word_index = tokenizer.word_index
tokenizer.fit_on_texts(texts)
# Convert the text data to a sequence of integer-encoded tokens
sequences = tokenizer.texts_to_sequences(texts)
# Pad the sequences to a maximum length of 20
max_length = 20
input_padded_sequences = pad_sequences(sequences, maxlen=max_length, padding='post')
print(tokenizer.sequences_to_texts(input_padded_sequences)[1])

# Define a list of text strings
texts2 = train['target']
# Create a tokenizer and fit on the text data
tokenizer2 = Tokenizer(num_words = vocab_size, oov_token=oov_tok)
word_index2 = tokenizer2.word_index
tokenizer2.fit_on_texts(texts2)
# Convert the text data to a sequence of integer-encoded tokens
sequences2 = tokenizer2.texts_to_sequences(texts2)
target_padded_sequences = pad_sequences(sequences2, maxlen=max_length, padding='post')
print(tokenizer.sequences_to_texts(target_padded_sequences)[1])

# Model

In [ ]:
import numpy as np
def decoder_input_data (text):
    # Define a 2D numpy array
    data = np.array(text)
    # Shift the data to the right by one position and replace first column with zeros
    shifted_data = np.roll(data, 1, axis=1)
    shifted_data[:, 0] = 0
    return shifted_data

In [ ]:
from keras.layers import Input, LSTM, Dense
from keras.models import Model
from sklearn.model_selection import train_test_split

latent_dim = 256
batch_size = 32 
epochs = 10 
input_dim = 20
output_dim = 20

encoder_input_data = input_padded_sequences
decoder_input_data  = decoder_input_data (input_padded_sequences)
decoder_target_data = target_padded_sequences

# Define the input sequence
encoder_inputs = Input(shape=(None, input_dim))

# Define the encoder LSTM
encoder_lstm = LSTM(latent_dim, return_state=True)

# Get the encoder outputs and states
encoder_outputs, state_h, state_c = encoder_lstm(encoder_inputs)


# Discard the encoder outputs and only keep the states
encoder_states = [state_h, state_c]

# Define the decoder inputs
decoder_inputs = Input(shape=(None, output_dim))

# Define the decoder LSTM
decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)

# Get the decoder outputs and states
decoder_outputs, _, _ = decoder_lstm(decoder_inputs, initial_state=encoder_states)

# Define the output layer
decoder_dense = Dense(output_dim, activation='softmax')

# Apply the output layer to the decoder outputs
decoder_outputs = decoder_dense(decoder_outputs)

# Define the model inputs and outputs
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

# Compile the model
model.compile(optimizer='rmsprop', loss='categorical_crossentropy')

In [ ]:
model.fit([encoder_input_data, decoder_input_data], decoder_target_data,
          batch_size=batch_size,
          epochs=epochs,
          validation_split=0.2)

# Translation

In [ ]:
def translate(sentence, encoder, decoder):
    
    attention_plot = np.zeros((max_length_targ, max_length_inp))
    sentence = preprocess_sentence(sentence)

    inputs = [inp_lang.word_index[i] for i in sentence.split(' ')]
    inputs = tf.keras.preprocessing.sequence.pad_sequences([inputs],maxlen=max_length_inp, padding='post')

    inputs = tf.convert_to_tensor(inputs)

    result = ''

    hidden = [tf.zeros((1,units))]
    enc_out, enc_hidden = encoder(inputs, hidden)

    dec_hidden = enc_hidden

    dec_input = tf.expand_dims([targ_lang.word_index['<start>']], 0)

    for t in range(max_length_targ):
        predictions, dec_hidden, attention_weights = decoder(dec_input, dec_hidden, enc_out)
        predicted_id = tf.argmax(predictions[0]).numpy()
        result += targ_lang.index_word[predicted_id] + ' '

    if targ_lang.index_word[predicted_id] == '<end>':
    return result, sentence
    
    dec_input = tf.expand_dims([predicted_id], 0)
return result, sentence